# Prediction Routine

## Imports

In [1]:
import os
import pandas as pd
from tensorflow.keras.models import model_from_json  # type: ignore

from deep.modelling.custom_loss import CategoricalFocalLoss
from deep.predict.model_fetcher import fetch_model
from deep.predict.loss_calculator import get_loss
from deep.preprocess.image_cleaner import clean_test_directory 
from deep.preprocess.resize_inputs import resize_test_album 
from deep.predict.predict_album import predict_from_directory
from deep.constants import MODEL_DICT, MODEL_IMAGE_SIZE, CLEANER_JSONS, MODEL_CONFIGS, SPLITTER_JSONS


2025-05-01 22:08:06.370367: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 22:08:06.371169: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-01 22:08:06.375309: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-01 22:08:06.386055: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746133686.405814   16465 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746133686.41

## Prediction Arguments

In [ ]:
# Pipeline args
image_dir = "Path/to/your/images" # provide a valid directory with images
cleaner_config_path = CLEANER_JSONS / "" # point to a valid cleaning json
split_config_path = SPLITTER_JSONS / "split_20250501T142850Z.json"

# Model initializing args
model_json_path = MODEL_CONFIGS / "model_expanded_architecture_2025-04-30.json"
model = input(f"Select a model from {list(MODEL_DICT.keys())}: ") # select one of our models

## Preprocessing 

This preprocessing schema adheres to the rule of no peeking, and modifies the provided data using only deterministic methods i.e. those learned from our training to show decent results.

In [ ]:
# Cleaner
clean_test_directory(config_path=cleaner_config_path, input_dir=image_dir)

# Resizer
resize_test_album(model=MODEL_DICT[model], path=image_dir)

## Loading the Model

We check if the model is present in the `prediction_pipeline` directory, otherwise we fetch it - consult constants.py and the README.md for this module, for more details.

In [3]:
# Get Model weights
weights_path = fetch_model(model=model)

# Get Loss from Split info 
loss = get_loss(split_config_path)

# Load the model architecture from the JSON file
with open(os.path.join(MODEL_CONFIGS, 'model_base_architecture_2025-04-30.json'), 'r') as json_file:
    model_json = json_file.read()

custom_objects = {"CategoricalFocalLoss": CategoricalFocalLoss}
model = model_from_json(model_json, custom_objects={'CategoricalFocalLoss': loss})

# Load the model weights
model.load_weights(weights_path)

Model already exists at ./BASE_MODEL_00.weights.h5, skipping download.


2025-05-01 14:32:14.061887: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Predict

Finally predictions are made on each image.

In [ ]:
# Make predictions
df_preds = predict_from_directory(model, image_dir, MODEL_IMAGE_SIZE[MODEL_DICT[model]])
df_preds.head()

In [ ]:
# Store the results
df_preds.to_csv()